In [2]:
# ============================================================
# SURFACE ROUGHNESS MINI-MODEL
#
# INPUT:
#   P, v, h, t
#
# OUTPUT:
#   Surface Roughness Ra (µm)
#
# Validation:
#   LOOCV
# ============================================================

import numpy as np
import pandas as pd
import warnings
import joblib

warnings.filterwarnings("ignore")


# ============================================================
# 1. EXPERIMENTAL DATA
# ============================================================

raw_data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    # --------------------------------------------------------
    # Replace these with the EXACT roughness measurements
    # from your experimental table if necessary.
    #
    # Values are Ra in micrometers.
    # --------------------------------------------------------
    "Roughness_Ra_um": [
        5.4,   # Row 1
        8.2,   # Row 2
        5.3,   # Row 3
        5.5,   # Row 4
        4.5,   # Row 5
        5.4,   # Row 6
        4.7,   # Row 7
        5.3,   # Row 8
        4.1,   # Row 9
        4.7,   # Row 10
        4.4,   # Row 11
        5.0,   # Row 12
        4.6,   # Row 13
        5.1,   # Row 14
        4.1,   # Row 15
        4.8,   # Row 16
        3.7,   # Row 17
        5.2,   # Row 18
        4.0,   # Row 19
        5.8    # Row 20
        ]
})


# ============================================================
# 2. AVERAGE REPEATED PROCESS CONDITIONS
# ============================================================

group_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    raw_data
    .groupby(group_cols, as_index=False)
    ["Roughness_Ra_um"]
    .mean()
)


print("=" * 70)
print("UNIQUE PROCESS-CONDITION DATASET")
print("=" * 70)

print(
    data.to_string(index=False)
)

print("\nOriginal measurements :", len(raw_data))
print("Unique conditions     :", len(data))


# ============================================================
# 3. INPUT / TARGET
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

target = "Roughness_Ra_um"

X = data[features]
y = data[target]


# ============================================================
# 4. IMPORT MODELS
# ============================================================

from sklearn.model_selection import LeaveOneOut, cross_val_predict

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 5. CANDIDATE MODELS
# ============================================================

models = {

    # --------------------------------------------------------
    # Linear Regression
    # --------------------------------------------------------
    "Linear Regression":
        LinearRegression(),


    # --------------------------------------------------------
    # Ridge
    # --------------------------------------------------------
    "Ridge":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),


    # --------------------------------------------------------
    # Lasso
    # --------------------------------------------------------
    "Lasso":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=0.01))
        ]),


    # --------------------------------------------------------
    # Polynomial Degree 2
    # --------------------------------------------------------
    "Polynomial Degree 2":
        Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),
            (
                "model",
                Ridge(alpha=10.0)
            )
        ]),


    # --------------------------------------------------------
    # Polynomial Degree 3
    # --------------------------------------------------------
    "Polynomial Degree 3 + Ridge":
        Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=3,
                    include_bias=False
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Ridge(alpha=10.0)
            )
        ]),


    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------
    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),


    # --------------------------------------------------------
    # Extra Trees
    # --------------------------------------------------------
    "Extra Trees":
        ExtraTreesRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),


    # --------------------------------------------------------
    # Gradient Boosting
    # --------------------------------------------------------
    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            min_samples_leaf=2,
            random_state=42
        )
}


# ============================================================
# 6. OPTIONAL XGBOOST
# ============================================================

try:

    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=42
    )

    print("\nXGBoost available.")

except ImportError:

    print("\nXGBoost not installed.")


# ============================================================
# 7. OPTIONAL CATBOOST
# ============================================================

try:

    from catboost import CatBoostRegressor

    models["CatBoost"] = CatBoostRegressor(
        iterations=200,
        depth=3,
        learning_rate=0.03,
        l2_leaf_reg=5,
        loss_function="RMSE",
        verbose=False,
        random_seed=42
    )

    print("CatBoost available.")

except ImportError:

    print("CatBoost not installed.")


# ============================================================
# 8. LOOCV MODEL COMPARISON
# ============================================================

loo = LeaveOneOut()

results = []
all_predictions = {}

for name, model in models.items():

    y_pred = cross_val_predict(
        model,
        X,
        y,
        cv=loo
    )

    rmse = np.sqrt(
        mean_squared_error(
            y,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y,
        y_pred
    )

    r2 = r2_score(
        y,
        y_pred
    )

    # Training R²
    model.fit(X, y)

    y_train_pred = model.predict(X)

    train_r2 = r2_score(
        y,
        y_train_pred
    )

    results.append({

        "Model": name,

        "LOOCV_RMSE_um":
            rmse,

        "LOOCV_MAE_um":
            mae,

        "LOOCV_R2":
            r2,

        "Training_R2":
            train_r2
    })

    all_predictions[name] = y_pred


# ============================================================
# 9. MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values(
        by="LOOCV_RMSE_um"
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 70)
print("ROUGHNESS MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 10. BEST MODEL
# ============================================================

best_name = results_df.iloc[0]["Model"]

best_model = models[best_name]

best_model.fit(X, y)

best_predictions = all_predictions[best_name]


print("\n")
print("=" * 70)
print("BEST ROUGHNESS MODEL")
print("=" * 70)

print(
    "Selected model:",
    best_name
)

print(
    f"LOOCV RMSE = "
    f"{results_df.iloc[0]['LOOCV_RMSE_um']:.4f} µm"
)

print(
    f"LOOCV MAE  = "
    f"{results_df.iloc[0]['LOOCV_MAE_um']:.4f} µm"
)

print(
    f"LOOCV R²   = "
    f"{results_df.iloc[0]['LOOCV_R2']:.4f}"
)

print(
    f"Training R² = "
    f"{results_df.iloc[0]['Training_R2']:.4f}"
)


# ============================================================
# 11. ACTUAL VS PREDICTED
# ============================================================

comparison = data.copy()

comparison["Predicted_Roughness_um"] = (
    best_predictions
)

comparison["Error_um"] = (
    comparison["Predicted_Roughness_um"]
    - comparison["Roughness_Ra_um"]
)

comparison["Absolute_Error_um"] = (
    comparison["Error_um"].abs()
)


print("\n")
print("=" * 70)
print("ACTUAL VS LOOCV PREDICTED")
print("=" * 70)

print(
    comparison[
        [
            "P_W",
            "v_mm_per_s",
            "h_mm",
            "t_mm",
            "Roughness_Ra_um",
            "Predicted_Roughness_um",
            "Error_um"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ============================================================
# 12. PHYSICS-INSPIRED FEATURE TEST
# ============================================================

data["VED_J_per_mm3"] = (
    data["P_W"]
    / (
        data["v_mm_per_s"]
        * data["h_mm"]
        * data["t_mm"]
    )
)

data["ED_J_per_mm2"] = (
    data["P_W"]
    / (
        data["v_mm_per_s"]
        * data["h_mm"]
    )
)

data["log_VED"] = np.log(
    data["VED_J_per_mm3"]
)


feature_sets = {

    "P_v_h_t": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm"
    ],

    "VED_only": [
        "VED_J_per_mm3"
    ],

    "P_v_h_t_VED": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm",
        "VED_J_per_mm3"
    ],

    "P_v_h_t_logVED": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm",
        "log_VED"
    ],

    "VED_ED": [
        "VED_J_per_mm3",
        "ED_J_per_mm2"
    ]
}


feature_results = []


for feature_name, feature_list in feature_sets.items():

    X_feature = data[feature_list]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    y_pred = cross_val_predict(
        model,
        X_feature,
        y,
        cv=LeaveOneOut()
    )

    rmse = np.sqrt(
        mean_squared_error(
            y,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y,
        y_pred
    )

    r2 = r2_score(
        y,
        y_pred
    )

    feature_results.append({

        "Feature_Set": feature_name,

        "LOOCV_RMSE_um":
            rmse,

        "LOOCV_MAE_um":
            mae,

        "LOOCV_R2":
            r2
    })


feature_results_df = (
    pd.DataFrame(feature_results)
    .sort_values(
        "LOOCV_RMSE_um"
    )
)


print("\n")
print("=" * 70)
print("PHYSICS-INSPIRED FEATURE COMPARISON")
print("=" * 70)

print(
    feature_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 13. FINAL FIT
# ============================================================

best_model.fit(
    data[features],
    y
)


# ============================================================
# 14. EXAMPLE PREDICTION
# ============================================================

example = pd.DataFrame({

    "P_W": [270],

    "v_mm_per_s": [1500],

    "h_mm": [0.08],

    "t_mm": [0.04]
})


predicted_roughness = (
    best_model.predict(example)[0]
)


print("\n")
print("=" * 70)
print("EXAMPLE ROUGHNESS PREDICTION")
print("=" * 70)

print(example)

print(
    f"\nPredicted Ra = "
    f"{predicted_roughness:.3f} µm"
)


# ============================================================
# 15. SAVE BEST MODEL
# ============================================================

joblib.dump(
    best_model,
    "Roughness_mini_model.pkl"
)

joblib.dump(
    features,
    "Roughness_feature_order.pkl"
)


print("\n")
print("Saved:")
print("  Roughness_mini_model.pkl")
print("  Roughness_feature_order.pkl")

UNIQUE PROCESS-CONDITION DATASET
 P_W  v_mm_per_s  h_mm  t_mm  Roughness_Ra_um
 250        1500 0.080  0.04         5.300000
 260        1450 0.075  0.04         4.600000
 260        1450 0.085  0.04         5.400000
 260        1550 0.075  0.04         5.400000
 260        1550 0.085  0.04         4.100000
 270        1400 0.080  0.04         4.700000
 270        1500 0.070  0.04         4.700000
 270        1500 0.080  0.04         4.833333
 270        1500 0.090  0.04         5.300000
 270        1600 0.080  0.04         4.500000
 280        1450 0.075  0.04         8.200000
 280        1450 0.085  0.04         5.000000
 280        1550 0.075  0.04         4.100000
 280        1550 0.085  0.04         4.400000
 290        1500 0.080  0.04         5.100000

Original measurements : 20
Unique conditions     : 15

XGBoost available.
CatBoost available.


ROUGHNESS MODEL COMPARISON
                      Model  LOOCV_RMSE_um  LOOCV_MAE_um  LOOCV_R2  Training_R2
                Extra Trees

In [3]:
# ============================================================
# ROUGHNESS SENSITIVITY ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    HuberRegressor
)

from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. START FROM THE ORIGINAL 20 MEASUREMENTS
# ============================================================

# IMPORTANT:
# This must contain your exact experimental values.

raw_data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "Roughness_Ra_um": [
        5.4, 8.2, 5.3, 5.5, 4.5,
        5.4, 4.7, 5.3, 4.1, 4.7,
        4.4, 5.0, 4.6, 5.1, 4.1,
        4.8, 3.7, 5.2, 4.0, 5.8
    ]
})


# ============================================================
# 2. CREATE UNIQUE PROCESS-CONDITION DATASET
# ============================================================

group_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    raw_data
    .groupby(group_cols, as_index=False)
    ["Roughness_Ra_um"]
    .mean()
)

print("\nUnique process conditions:", len(data))


# ============================================================
# 3. CREATE ENERGY FEATURES
# ============================================================

data["VED_J_per_mm3"] = (
    data["P_W"]
    / (
        data["v_mm_per_s"]
        * data["h_mm"]
        * data["t_mm"]
    )
)

data["ED_J_per_mm2"] = (
    data["P_W"]
    / (
        data["v_mm_per_s"]
        * data["h_mm"]
    )
)

data["log_VED"] = np.log(
    data["VED_J_per_mm3"]
)


# ============================================================
# 4. COMMON SETTINGS
# ============================================================

loo = LeaveOneOut()

results = []


# ============================================================
# FUNCTION FOR EVALUATION
# ============================================================

def evaluate_model(
    name,
    X,
    y,
    model,
    log_target=False
):

    if log_target:

        y_model = np.log(y)

        pred_log = cross_val_predict(
            model,
            X,
            y_model,
            cv=loo
        )

        predictions = np.exp(
            pred_log
        )

    else:

        predictions = cross_val_predict(
            model,
            X,
            y,
            cv=loo
        )

    rmse = np.sqrt(
        mean_squared_error(
            y,
            predictions
        )
    )

    mae = mean_absolute_error(
        y,
        predictions
    )

    r2 = r2_score(
        y,
        predictions
    )

    results.append({

        "Model": name,

        "RMSE_um": rmse,

        "MAE_um": mae,

        "R2": r2
    })

    return predictions


# ============================================================
# 5. TEST 1 — STANDARD EXTRA TREES
# ============================================================

X = data[
    [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm"
    ]
]

y = data[
    "Roughness_Ra_um"
]


evaluate_model(

    "Extra Trees - P,v,h,t",

    X,
    y,

    ExtraTreesRegressor(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42
    )
)


# ============================================================
# 6. TEST 2 — VED ONLY
# ============================================================

X = data[
    [
        "VED_J_per_mm3"
    ]
]


evaluate_model(

    "Ridge - VED",

    X,
    y,

    Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=1.0)
        )
    ])
)


# ============================================================
# 7. TEST 3 — LOG(VED)
# ============================================================

X = data[
    [
        "log_VED"
    ]
]


evaluate_model(

    "Ridge - log(VED)",

    X,
    y,

    Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=1.0)
        )
    ])
)


# ============================================================
# 8. TEST 4 — LOG TARGET
# ============================================================

X = data[
    [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm"
    ]
]


evaluate_model(

    "Ridge - log(Ra)",

    X,
    y,

    Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=1.0)
        )
    ]),

    log_target=True
)


# ============================================================
# 9. TEST 5 — ROBUST REGRESSION
# ============================================================

evaluate_model(

    "Huber - P,v,h,t",

    X,
    y,

    Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "huber",
            HuberRegressor(
                epsilon=1.35,
                max_iter=500
            )
        )
    ])
)


# ============================================================
# 10. TEST 6 — POLYNOMIAL + RIDGE
# ============================================================

evaluate_model(

    "Polynomial 2 + Ridge",

    X,
    y,

    Pipeline([
        (
            "poly",
            PolynomialFeatures(
                degree=2,
                include_bias=False
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=10.0)
        )
    ])
)


# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values(
        "RMSE_um"
    )
    .reset_index(drop=True)
)

print("\n")
print("=" * 70)
print("ROUGHNESS SENSITIVITY RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 12. OUTLIER SENSITIVITY
# ============================================================

print("\n")
print("=" * 70)
print("8.2 µm OUTLIER SENSITIVITY")
print("=" * 70)

# Locate the 8.2 µm experimental measurement
outlier_mask = (
    raw_data["Roughness_Ra_um"] == 8.2
)

print(
    raw_data[outlier_mask]
)


# Remove ONLY for sensitivity analysis.
# DO NOT delete from the real dataset permanently.

sensitivity_raw = raw_data[
    ~outlier_mask
].copy()


sensitivity_data = (
    sensitivity_raw
    .groupby(
        group_cols,
        as_index=False
    )
    ["Roughness_Ra_um"]
    .mean()
)


X_sens = sensitivity_data[
    [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm"
    ]
]

y_sens = sensitivity_data[
    "Roughness_Ra_um"
]


# ------------------------------------------------------------
# Extra Trees without 8.2 µm point
# ------------------------------------------------------------

model_sens = ExtraTreesRegressor(

    n_estimators=300,

    max_depth=3,

    min_samples_leaf=2,

    max_features=0.8,

    random_state=42
)


pred_sens = cross_val_predict(

    model_sens,

    X_sens,

    y_sens,

    cv=LeaveOneOut()
)


rmse_sens = np.sqrt(
    mean_squared_error(
        y_sens,
        pred_sens
    )
)

mae_sens = mean_absolute_error(
    y_sens,
    pred_sens
)

r2_sens = r2_score(
    y_sens,
    pred_sens
)


print("\nWITH 8.2 µm:")
print(
    f"RMSE = "
    f"{results_df.iloc[0]['RMSE_um']:.4f} µm"
)

print(
    f"MAE  = "
    f"{results_df.iloc[0]['MAE_um']:.4f} µm"
)

print(
    f"R²   = "
    f"{results_df.iloc[0]['R2']:.4f}"
)


print("\nWITHOUT 8.2 µm:")
print(
    f"RMSE = "
    f"{rmse_sens:.4f} µm"
)

print(
    f"MAE  = "
    f"{mae_sens:.4f} µm"
)

print(
    f"R²   = "
    f"{r2_sens:.4f}"
)


# ============================================================
# 13. IDENTIFY THE OUTLIER'S PROCESS CONDITION
# ============================================================

print("\n")
print("=" * 70)
print("POTENTIAL OUTLIER")
print("=" * 70)

print(
    raw_data[
        raw_data["Roughness_Ra_um"] == 8.2
    ].to_string(index=False)
)


Unique process conditions: 15


ROUGHNESS SENSITIVITY RESULTS
                Model  RMSE_um  MAE_um      R2
Extra Trees - P,v,h,t   1.0738  0.7316 -0.2983
      Huber - P,v,h,t   1.0823  0.6852 -0.3191
     Ridge - log(VED)   1.1079  0.7795 -0.3821
      Ridge - log(Ra)   1.1080  0.7718 -0.3825
          Ridge - VED   1.1144  0.7831 -0.3984
 Polynomial 2 + Ridge   1.1166  0.8007 -0.4040


8.2 µm OUTLIER SENSITIVITY
   P_W  v_mm_per_s   h_mm  t_mm  Roughness_Ra_um
1  280        1450  0.075  0.04              8.2

WITH 8.2 µm:
RMSE = 1.0738 µm
MAE  = 0.7316 µm
R²   = -0.2983

WITHOUT 8.2 µm:
RMSE = 0.4906 µm
MAE  = 0.4198 µm
R²   = -0.2777


POTENTIAL OUTLIER
 P_W  v_mm_per_s  h_mm  t_mm  Roughness_Ra_um
 280        1450 0.075  0.04              8.2
